<a href="https://colab.research.google.com/github/BU-Spark/ds-ciss-predictive-homlessness/blob/data-cleaning-feature/Fix_Rent_%2B_Income.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
base_path = '/content/drive/My Drive/DS539'
rent_path = f'{base_path}/ACS Rent'
population_path = f'{base_path}/ACS Population'
income_path = f'{base_path}/Income ACS'

tract2019_path = f'{base_path}/tract_coc_match_2019 - tract_coc_match_2019.csv'
tract2022_path = f'{base_path}/tract_coc_match_2022 - tract_coc_match_2022.csv'


In [46]:
import pandas as pd

tract2019 = pd.read_csv(tract2019_path)
tract2022 = pd.read_csv(tract2022_path)


In [47]:
import os

# Function to load data for a given year
def load_data(year):
    # Use 2010 data for 2007–2009
    source_year = year if year >= 2010 else 2010

    # Define the file names
    pop_file = f"ACSDT5Y{source_year}.B01003-Data.csv"
    income_file = f"ACSDT5Y{source_year}.B19013-Data.csv"
    rent_file = f"ACSDT5Y{source_year}.B25064-Data.csv"

    # Full paths to the CSV files
    pop_path = os.path.join(population_path, pop_file)
    income_path_full = os.path.join(income_path, income_file)
    rent_path_full = os.path.join(rent_path, rent_file)

    # Check if files exist
    if not os.path.exists(pop_path):
        print(f"Population data for year {source_year} not found.")
        return None
    if not os.path.exists(income_path_full):
        print(f"Income data for year {source_year} not found.")
        return None
    if not os.path.exists(rent_path_full):
        print(f"Rent data for year {source_year} not found.")
        return None

    try:
        # Load the datasets
        pop = pd.read_csv(pop_path)
        income = pd.read_csv(income_path_full)
        rent = pd.read_csv(rent_path_full)

        # Merge the datasets on GEO_ID
        df = pop.merge(income, on="GEO_ID").merge(rent, on="GEO_ID")

        # Assign the *target* year (not just the source year)
        df['year'] = year

        return df

    except Exception as e:
        print(f"Error loading data for year {year}: {e}")
        return None

# Load and concatenate all years from 2007 to 2023
years = list(range(2007, 2024))
all_data = pd.concat([load_data(year) for year in years if load_data(year) is not None], ignore_index=True)

# Show preview
print(all_data.head())


                 GEO_ID                                     NAME_x  \
0             Geography                       Geographic Area Name   
1  1400000US01001020100  Census Tract 201, Autauga County, Alabama   
2  1400000US01001020200  Census Tract 202, Autauga County, Alabama   
3  1400000US01001020300  Census Tract 203, Autauga County, Alabama   
4  1400000US01001020400  Census Tract 204, Autauga County, Alabama   

              B01003_001M      B01003_001E  Unnamed: 4_x  \
0  Margin of Error!!Total  Estimate!!Total           NaN   
1                     155             1809           NaN   
2                     258             2020           NaN   
3                     332             3543           NaN   
4                     260             4840           NaN   

                                      NAME_y  \
0                       Geographic Area Name   
1  Census Tract 201, Autauga County, Alabama   
2  Census Tract 202, Autauga County, Alabama   
3  Census Tract 203, Autau

In [48]:
output_path = '/content/drive/My Drive/DS539/all_data_combined.csv'
all_data.to_csv(output_path, index=False)

In [49]:
filtered_data = all_data[['GEO_ID', 'B25064_001E', 'B19013_001E', 'B01003_001E', 'year']]

# Display the filtered data
print(filtered_data.head())

                 GEO_ID                  B25064_001E  \
0             Geography  Estimate!!Median gross rent   
1  1400000US01001020100                          680   
2  1400000US01001020200                          584   
3  1400000US01001020300                          733   
4  1400000US01001020400                          954   

                                         B19013_001E      B01003_001E  year  
0  Estimate!!Median household income in the past ...  Estimate!!Total  2007  
1                                              70222             1809  2007  
2                                              41091             2020  2007  
3                                              44031             3543  2007  
4                                              56627             4840  2007  


In [50]:
filtered_path = '/content/drive/My Drive/DS539/filtered_data.csv'
filtered_data.to_csv(filtered_path, index=False)

In [51]:
def pad_geoid(geoid):
    return str(geoid).zfill(11)  # Ensure GEOID is a string and then pad to 11 digits

# Apply the padding function to the GEOID column in both crosswalk datasets
tract2019['GEOID'] = tract2019['GEOID'].apply(pad_geoid)
tract2022['GEOID'] = tract2022['GEOID'].apply(pad_geoid)

In [52]:
filtered_data['GEO_ID'] = filtered_data['GEO_ID'].astype(str).str[-11:]

<ipython-input-52-bfc50e08eb1a>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['GEO_ID'] = filtered_data['GEO_ID'].astype(str).str[-11:]


In [53]:
filtered_data.head()

,GEO_ID,B25064_001E,B19013_001E,B01003_001E,year
0,Geography,Estimate!!Median gross rent,Estimate!!Median household income in the past ...,Estimate!!Total,2007
1,01001020100,680,70222,1809,2007
2,01001020200,584,41091,2020,2007
3,01001020300,733,44031,3543,2007
4,01001020400,954,56627,4840,2007


In [54]:
# Create clean copies of the split data
data_2007_2019 = filtered_data[filtered_data['year'] <= 2019].copy()
data_2020_2023 = filtered_data[filtered_data['year'] >= 2020].copy()

# Merge with correct crosswalk
data_2007_2019 = data_2007_2019.merge(
    tract2019[['GEOID', 'COCNUM']],
    left_on='GEO_ID',
    right_on='GEOID',
    how='left'
)

data_2020_2023 = data_2020_2023.merge(
    tract2022[['GEOID', 'COCNUM']],
    left_on='GEO_ID',
    right_on='GEOID',
    how='left'
)

# Drop right-side GEOID (since we have GEO_ID already)
data_2007_2019.drop(columns=['GEOID'], inplace=True)
data_2020_2023.drop(columns=['GEOID'], inplace=True)



In [55]:
final_data = pd.concat([data_2007_2019, data_2020_2023], ignore_index=True)


output_path = '/content/drive/My Drive/DS539/final_data.csv'
final_data.to_csv(output_path, index=False)


In [56]:
final_data = final_data.rename(columns={
    'B25064_001E': 'median rent',
    'B19013_001E': 'median income',
    'B01003_001E': 'total population'
})

# Verify the column renaming
print(final_data.head())

        GEO_ID                  median rent  \
0    Geography  Estimate!!Median gross rent   
1  01001020100                          680   
2  01001020200                          584   
3  01001020300                          733   
4  01001020400                          954   

                                       median income total population  year  \
0  Estimate!!Median household income in the past ...  Estimate!!Total  2007   
1                                              70222             1809  2007   
2                                              41091             2020  2007   
3                                              44031             3543  2007   
4                                              56627             4840  2007   

   COCNUM  
0     NaN  
1  AL-504  
2  AL-504  
3  AL-504  
4  AL-504  


In [57]:
final_data['median income'] = pd.to_numeric(final_data['median income'], errors='coerce')
final_data['median rent'] = pd.to_numeric(final_data['median rent'], errors='coerce')
final_data['total population'] = pd.to_numeric(final_data['total population'], errors='coerce')

# Group by both COCNUM and year
weighted_avg = final_data.groupby(['COCNUM', 'year']).apply(
    lambda x: pd.Series({
        'weighted median income': (x['median income'] * x['total population']).sum() / x['total population'].sum(),
        'weighted median rent': (x['median rent'] * x['total population']).sum() / x['total population'].sum(),
        'total population': x['total population'].sum(),
        'tract count': len(x)
    })
).reset_index()

# Preview
print(weighted_avg.head())


   COCNUM  year  weighted median income  weighted median rent  \
0  AK-500  2007            78328.533122           1062.638009   
1  AK-500  2008            78328.533122           1062.638009   
2  AK-500  2009            78328.533122           1062.638009   
3  AK-500  2010            78328.533122           1062.638009   
4  AK-500  2011            80808.699781           1037.436470   

   total population  tract count  
0          284267.0         55.0  
1          284267.0         55.0  
2          284267.0         55.0  
3          284267.0         55.0  
4          287390.0         55.0  


<ipython-input-57-7401aa36cb9d>:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_avg = final_data.groupby(['COCNUM', 'year']).apply(


In [58]:
output_path = '/content/drive/My Drive/DS539/corrected_stuff.csv'
weighted_avg.to_csv(output_path, index=False)

In [59]:
import pandas as pd

# Step 2: Calculate the descriptive statistics for the relevant columns
summary_stats = weighted_avg[['weighted median income', 'weighted median rent', 'total population']].describe()

# Calculate the interquartile range (IQR) for each column manually (Q3 - Q1)
iqr = weighted_avg[['weighted median income', 'weighted median rent', 'total population']].quantile(0.75) - weighted_avg[['weighted median income', 'weighted median rent', 'total population']].quantile(0.25)

# Step 3: Manually calculate the median for each column
median = weighted_avg[['weighted median income', 'weighted median rent', 'total population']].median()

# Combine summary statistics, IQR, and median into a single DataFrame
summary_stats.loc['IQR'] = iqr
summary_stats.loc['median'] = median

# Step 4: Round all the statistics to the nearest whole number
summary_stats = summary_stats.round(0)


# Step 6: Display the rounded summary statistics
print(summary_stats)


        weighted median income  weighted median rent  total population
count                   6646.0                6646.0            6646.0
mean                   63734.0                1004.0          815264.0
std                    19238.0                 304.0         1159451.0
min                    18437.0                 425.0           29235.0
25%                    49911.0                 786.0          261697.0
50%                    59206.0                 949.0          486123.0
75%                    73429.0                1146.0          896482.0
max                   159214.0                2579.0        11500012.0
IQR                    23519.0                 360.0          634785.0
median                 59206.0                 949.0          486123.0


In [60]:
max_row = weighted_avg.loc[weighted_avg['total population'].idxmax()]
print(max_row)

COCNUM                          TX-607
year                              2023
weighted median income    77357.742119
weighted median rent       1157.301902
total population            11500012.0
tract count                     2872.0
Name: 5965, dtype: object


In [61]:
filtered_row = weighted_avg[(weighted_avg['COCNUM'] == 'TX-607') & (weighted_avg['year'] == 2019)]

# Display the result
print(filtered_row)

      COCNUM  year  weighted median income  weighted median rent  \
5961  TX-607  2019            61361.651046            964.263052   

      total population  tract count  
5961        10916292.0       2167.0  
